# Conversion Musique > 16 Bit > 44.1 KHz

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from mutagen.flac import FLAC
from tqdm.auto import tqdm

def analyser_et_convertir(arguments: tuple) -> str:
    """
    Worker parallèle. Analyse les métadonnées et lance FFmpeg si nécessaire.
    """
    chemin_str, chemin_ffmpeg = arguments
    chemin_fichier = Path(chemin_str)
    chemin_temp = chemin_fichier.with_suffix('.temp.flac')
    
    try:
        # Extraire les propriétés audio sans charger le fichier complet
        audio = FLAC(chemin_fichier)
        
        # Identifier les fichiers haute résolution
        if audio.info.sample_rate > 44100 or audio.info.bits_per_sample > 16:
            
            # Paramètres de conversion sans perte de métadonnées
            commande = [
                chemin_ffmpeg,
                '-i', chemin_str,
                '-map', '0',                    # Conserver tous les flux (audio, vidéo/pochette)
                '-c:v', 'copy',                 # Ne pas altérer l'image
                '-c:a', 'flac',                 # Codec FLAC
                '-sample_fmt', 's16',           # Forcer la quantification 16 bits
                '-ar', '44100',                 # Forcer l'échantillonnage 44.1 kHz
                '-compression_level', '5',      # Ratio vitesse/compression standard
                '-dither_method', 'shibata',    # Algorithme de tramage qualitatif
                '-map_metadata', '0',           # Conserver les tags ID3/Vorbis
                '-loglevel', 'error',           # Masquer la sortie console FFmpeg
                '-y',                           # Écraser sans confirmation
                str(chemin_temp)
            ]
            
            # Exécuter l'encodeur
            subprocess.run(commande, check=True)
            
            # Transférer les attributs système (date de création/modification)
            shutil.copystat(chemin_fichier, chemin_temp)
            
            # Remplacer le fichier original par la version convertie
            chemin_temp.replace(chemin_fichier)
            
        return f"SUCCES|{chemin_str}"

    except KeyboardInterrupt:
        # Nettoyer immédiatement en cas d'arrêt manuel
        if chemin_temp.exists():
            chemin_temp.unlink(missing_ok=True)
        return "ANNULATION"

    except Exception as e:
        # Nettoyer et formater le message d'erreur
        if chemin_temp.exists():
            chemin_temp.unlink(missing_ok=True)
        return f"ERREUR|{chemin_str}|{str(e)}"


def optimiser_bibliotheque_flac(chemin_dossier: str, chemin_ffmpeg: str) -> None:
    """
    Gère l'arborescence, la reprise rapide et la distribution des tâches sur le processeur.
    """
    racine = Path(chemin_dossier)
    dossier_execution = Path.cwd()
    
    # Définir les chemins des journaux dans le dossier courant du carnet
    log_succes = dossier_execution / "conversion_succes.log"
    log_erreurs = dossier_execution / "conversion_erreurs.log"
    
    # 1. Purge des résidus d'une session précédente
    for fichier_temp in racine.rglob('*.temp.flac'):
        fichier_temp.unlink(missing_ok=True)
        
    # 2. Chargement de l'historique pour la reprise rapide
    fichiers_traites = set()
    if log_succes.exists():
        with open(log_succes, 'r', encoding='utf-8') as f:
            fichiers_traites = set(f.read().splitlines())

    # 3. Inventaire des fichiers à traiter
    taches_a_traiter = []
    for chemin_fichier in racine.rglob('*.flac'):
        chemin_str = str(chemin_fichier)
        if chemin_str not in fichiers_traites:
            taches_a_traiter.append((chemin_str, chemin_ffmpeg))

    if not taches_a_traiter:
        return

    # 4. Configuration du traitement parallèle (conserver 1 cœur pour le système)
    nb_threads = max(1, os.cpu_count() - 1)

    try:
        # 5. Exécution avec barre de progression
        with ThreadPoolExecutor(max_workers=nb_threads) as executeur:
            # Soumettre toutes les tâches
            futures = [executeur.submit(analyser_et_convertir, tache) for tache in taches_a_traiter]
            
            # Suivre l'avancement au fur et à mesure des terminaisons
            for future in tqdm(as_completed(futures), total=len(taches_a_traiter), desc="Progression globale", unit="piste"):
                resultat = future.result()
                
                if resultat == "ANNULATION":
                    continue
                
                parties = resultat.split('|')
                etat = parties[0]
                
                # Inscription conditionnelle dans les journaux
                if etat == "SUCCES":
                    chemin = parties[1]
                    with open(log_succes, 'a', encoding='utf-8') as f:
                        f.write(f"{chemin}\n")
                        
                elif etat == "ERREUR":
                    chemin = parties[1]
                    message_erreur = parties[2]
                    with open(log_erreurs, 'a', encoding='utf-8') as f:
                        f.write(f"Erreur sur {chemin} : {message_erreur}\n")
                        
    except KeyboardInterrupt:
        # Intercepter l'arrêt global pour stopper le processus proprement
        executeur.shutdown(wait=False, cancel_futures=True)


# Déclenchement de la fonction

In [ ]:
# Définition des chemins absolus (utiliser le préfixe 'r' pour Windows)
CHEMIN_BIBLIOTHEQUE = r"C:\Users\FLORIAN\Music\__QBDLX_Download"

# Remplacer par le chemin absolu exact de l'exécutable localisé précédemment via where.exe
CHEMIN_EXECUTABLE_FFMPEG = r"C:\Users\FLORIAN\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin\ffmpeg.exe"


optimiser_bibliotheque_flac(CHEMIN_BIBLIOTHEQUE, CHEMIN_EXECUTABLE_FFMPEG)

Progression globale: 100%|██████████| 17408/17408 [00:28<00:00, 620.51piste/s]


In [8]:
# Définition des chemins absolus (utiliser le préfixe 'r' pour Windows)
CHEMIN_BIBLIOTHEQUE = r"M:\musiques"

# Remplacer par le chemin absolu exact de l'exécutable localisé précédemment via where.exe
CHEMIN_EXECUTABLE_FFMPEG = r"C:\Users\FLORIAN\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin\ffmpeg.exe"


optimiser_bibliotheque_flac(CHEMIN_BIBLIOTHEQUE, CHEMIN_EXECUTABLE_FFMPEG)

Progression globale: 100%|██████████| 92600/92600 [2:13:37<00:00, 11.55piste/s]   
